In [16]:
#@title Run to install and load necessary modules
!pip install py3Dmol
import ipywidgets as widgets
import pandas as pd
import py3Dmol
from ipywidgets import Layout
import numpy as np

def HighlightResList(reslist, mutlist, sepreslist):
    reslist2 = [str(i) for i in reslist]
    mutlist2 = [str(i) for i in mutlist]
    if sepreslist:
        sepreslist2 = [str(i) for i in sepreslist]
    else:
        sepreslist2 = []
    triadlist = ['105','160','173']
    i = 0
    print(mutlist2)
    view.setStyle({'model': -1}, {"cartoon": {'color': 'blue'}})

    view.addStyle({'resi':reslist2},
              {'stick': {'color':'yellow'}})
    view.addStyle({'resi':triadlist},                    ###  Something different than red
              {'stick': {'color':'red'}})
    if sepreslist:
        view.addStyle({'resi':sepreslist2},
              {'stick': {'color':'green'}})
    view.zoomTo()
    view.show()
    return reslist2, triadlist , sepreslist2
style = {'description_width': 'initial'}

SizeChange = widgets.SelectMultiple(
    options = ['Larger Residue', 'Smaller Residue', 'No Change'],
    value = ['Larger Residue', 'Smaller Residue', 'No Change'],
    description="How does the new residue compare to original residue in size",
    disabled=False,
    style=style,
    layout = Layout(width='60%', height='60px')
)

HydrophobicChange = widgets.SelectMultiple(
    options = ['More Hydrophobic', 'Less Hydrophobic', 'No Change'],
    value = ['More Hydrophobic', 'Less Hydrophobic', 'No Change'],
    description="How does the new residue compare to original residue in hydrophobicity",
    disabled=False,
    style=style,
    layout = Layout(width='60%', height='60px')
)

TriadDistance = widgets.IntRangeSlider(
    value=[0,10],
    min=0,
    max=30,
    step=1,
    description="CA-CA Distance to catalytic serine",
    style=style,
    layout = Layout(width='60%', height='60px')
)

ChargeChange = widgets.SelectMultiple(
    options = ['Positive Delta', 'Negative Delta', 'No Change'],
    value = ['Positive Delta', 'Negative Delta', 'No Change'],
    description="How does the new residue compare to original residue in terms of charge",
    disabled=False,
    style=style,
    layout = Layout(width='60%', height='60px')
)

HBondChange = widgets.SelectMultiple(
    options = ['Gain HBonds', 'Lose HBonds', 'No Change'],
    value = ['Gain HBonds','Lose HBonds','No Change'],
    description="How does the new residue compare to original residue in hydrogen bonding capability",
    disabled=False,
    style=style,
    layout = Layout(width='60%', height='60px')
)

AppCount = widgets.IntSlider(
    value=0,
    min=0,
    max=50,
    description="Minimum amount of times the mutation should appear in the set",
    disabled=False,
    readout=True,
    readout_format='d',
    style=style,
    layout = Layout(width='60%', height='60px')
)

Frequency = widgets.BoundedFloatText(
    value=0.0,
    min=0,
    max=100,
    step=0.1,
    description="Minimum frequency of the mutation in the set",
    disabled=False,
    readout=True,
    readout_format='.1f',
    style=style,
    layout = Layout(width='60%', height='60px')
)

PrintPymol = widgets.Checkbox(
    value=False,
    description= 'Show the PyMol command to recreate image',
    disabled=False,
    button_stlye = '',
    tooltip='Description',
    icon='check',
    layout = Layout(width='60%', height='60px')

)

ColorSpecRes = widgets.Text(
    value = '',
    placeholder  = 'Positions to display',
    description = 'Enter comma-separated list of positions to display',
    disabled=False,
    style=style,
    layout = Layout(width='60%', height='60px')
)



def HighlightMut(countlimit, sepreslist,size,hydro,triad, charge, hbond, pymolcmd):
    df2 = df[(df.FreqPerc >= countlimit) & (df.SizeDelta.isin(size)) & (df.TriadDist >= triad[0]) & (df.TriadDist <= triad[1]) & (df.HydroDelta.isin(hydro)) & (df.ChargeDelta.isin(charge)) & (df.HBondDelta.isin(hbond))]
    reslist = df2['Position'].tolist()
    mutlist = df2['Mutation'].tolist()
    poslist = []
    for mut in mutlist:
        pos = int(mut[1:-1])
        pos -= 1
        poslist.append(pos)
    if sepreslist:
        sepreslist2 = sepreslist.split(',')
        sepreslist3 = [int(i)-1 for i in sepreslist2 if i]
    else:
        sepreslist3 = []
    mutlist2 , triadlist, addmutlist = HighlightResList(poslist, mutlist, sepreslist3)
    if pymolcmd:
        triadcmd = ''
        mutcmd = ''
        addcmd = ''
        for res in triadlist:
            triadcmd += res+'+'
        for res in mutlist2:
            mutcmd += res+'+'
        for res in addmutlist:
            addcmd += res+'+'
        print('load %s; as cartoon; color blue; bg_color white; sele resi %s; show sticks, sele; color red, sele; sele resi %s; show sticks, sele; color yellow, sele' % (PDBFile, triadcmd, mutcmd))




Import your files into google colab, including the protein structure PDB file and the list of your mutations. Type the names of each file below, including their file extensions

In [17]:
# PDBFile = '4oyy.pdb'
# MutationFile = 'C3R1SR1G1_C3R2SR1G1_muts.csv'
PDBFile = '4fnkA.pdb'
MutationFile = 'fakedata_4fnkA_muts.csv'

In [18]:
#@title Run to load your protein
with open(PDBFile) as ifile:
    system = "".join([x for x in ifile])
view = py3Dmol.view(width=400, height=300)
view.addModelsAsFrames(system)
view.setStyle({'model': -1}, {"cartoon": {'color': 'spectrum'}})
view.zoomTo()
view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [20]:
#@title Run to initiate visualization
df = pd.read_csv(MutationFile)
widgets.interactive(HighlightMut, sepreslist=ColorSpecRes, hydro=HydrophobicChange,  countlimit = Frequency, size = SizeChange,  triad = TriadDistance, charge = ChargeChange, hbond = HBondChange,  pymolcmd = PrintPymol, )


interactive(children=(BoundedFloatText(value=80.0, description='Minimum frequency of the mutation in the set',…